In [2]:
import requests
import pandas as pd
import os
from datetime import datetime

from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser
from urllib.parse import urljoin

In [3]:
# Definisi URL
BASE_URL = os.getenv("URL_WEB")
ROBOTS_URL = urljoin(BASE_URL, "/robots.txt")

# Siapkan headers membuka menggunakan browser
headers = {
    "User-Agent": "Mozilla/5.0"
}

# Kirim request mengakses halaman robots.txt
response_robots = requests.get(
    ROBOTS_URL,
    headers=headers,
    timeout=30
)

# Tampilkan hasilnya
print("Status Code:", response_robots.status_code)
print("Isi robots.txt:")
print(response_robots.text)

Status Code: 200
Isi robots.txt:
User-Agent: *
Allow: /
Disallow: /private/

Sitemap: https://data.go.id/sitemap.xml



In [4]:
# Tarhet halaman yang ingin di scrapping
response = requests.get(
        BASE_URL+"/dataset",
        headers=headers,
        timeout=30,
        params={
            'q':"budaya",
            'perPage':100
        }
    )

response.raise_for_status()

html = response.text

print("Status Code:", response.status_code)
# print("Panjang HTML:", len(html))
soup = BeautifulSoup(html, "lxml")

print("Judul Website:", soup.title.get_text(strip=True))

print("\nPreview HTML:")
print(soup.prettify()[:1000])

Status Code: 200
Judul Website: Portal Satu Data Indonesia - Dataset

Preview HTML:
<!DOCTYPE html>
<html lang="en">
 <head>
  <meta charset="utf-8"/>
  <meta content="width=device-width, initial-scale=1" name="viewport"/>
  <link data-precedence="next" href="/_next/static/css/c6bfecd89233120c.css" rel="stylesheet"/>
  <link data-precedence="next" href="/_next/static/css/1950b75b695126f3.css" rel="stylesheet"/>
  <link as="script" fetchpriority="low" href="/_next/static/chunks/webpack-e1b92c0e148d9c88.js" rel="preload"/>
  <script async="" src="/_next/static/chunks/4bd1b696-cb40bdd1e1e80e9f.js">
  </script>
  <script async="" src="/_next/static/chunks/1684-fc95d9adc57d6b4a.js">
  </script>
  <script async="" src="/_next/static/chunks/main-app-880bf1748d113d6f.js">
  </script>
  <script async="" src="/_next/static/chunks/7535-61260e83c0bcc273.js">
  </script>
  <script async="" src="/_next/static/chunks/7947-ebc33f879badb291.js">
  </script>
  <script async="" src="/_next/static/chunks/

In [5]:
links = soup.find_all("a", href=True)

for link in links:
    href = link["href"]
    # Mengambil judul dari tag h3
    h3 = link.find("h3")
    # Mengambil deskripsi dari div dengan class text-muted-foreground
    deskripsi_div = link.find("div", class_="text-muted-foreground")
    if h3:
        title = h3.get_text(" ", strip=True)

        # Jika deskripsi ditemukan, ambil teksnya
        deskripsi = deskripsi_div.get_text(" ", strip=True) if deskripsi_div else ""

        if "/dataset/" in href and title:
            print("Judul:", title)
            print("Deskripsi:", deskripsi)
            print("URL:", urljoin(BASE_URL, href))
            print("-" * 50)

Judul: Indeks Budaya Hukum
Deskripsi: Indeks Budaya Hukum adalah salah satu pilar dalam Indeks Pembangunan Hukum (IPH) yang bertujuan untuk mengukur tingkat pemahaman, kesadaran, hingga kepatutan hukum masyarakat; serta mengukur tingkat kepatutan hukum lembaga hukum.
URL: https://data.go.id/dataset/dataset/indeks-budaya-hukum
--------------------------------------------------
Judul: Indeks Pembangunan Hukum
Deskripsi: Indeks Pembangunan Hukum (IPH) adalah indeks yang mengukur pilar pembangunan hukum sebagai representasi capaian pembangunan hukum di Indonesia. Adapun lima pilar dalam IPH terdiri dari budaya hukum (legal culture), materi hukum (legal substance), kel...
URL: https://data.go.id/dataset/dataset/indeks-pembangunan-hukum1
--------------------------------------------------
Judul: Indeks Pembangunan Kawasan Perdesaan
Deskripsi: Angka yang menggambarkan tingkat kemajuan dan perkembangan kawasan 
perdesaan yang ditinjau dari dimensi ekonomi, sosial budaya, lingkungan, jaringan pr

In [6]:
data = []

for link in soup.find_all("a", href=True):
    href = link["href"]

    # Mengambil judul dari tag h3
    h3 = link.find("h3")
    title = h3.get_text(" ", strip=True) if h3 else ""

    # Mengambil deskripsi dari div text-muted-foreground
    deskripsi_div = link.find("div", class_="text-muted-foreground")
    deskripsi = deskripsi_div.get_text(" ", strip=True) if deskripsi_div else ""

    if "/dataset/" in href and title:
        data.append({
            "judul_dataset": title,
            "deskripsi": deskripsi,
            "url_dataset": urljoin(BASE_URL, href)
        })

df = pd.DataFrame(data).drop_duplicates(subset=["url_dataset"])

display(df)

,judul_dataset,deskripsi,url_dataset
0,Indeks Budaya Hukum,Indeks Budaya Hukum adalah salah satu pilar da...,https://data.go.id/dataset/dataset/indeks-buda...
1,Indeks Pembangunan Hukum,Indeks Pembangunan Hukum (IPH) adalah indeks y...,https://data.go.id/dataset/dataset/indeks-pemb...
2,Indeks Pembangunan Kawasan Perdesaan,Angka yang menggambarkan tingkat kemajuan dan ...,https://data.go.id/dataset/dataset/indeks-pemb...
3,Data Organisasi Seni di Kabupaten Mukomuko Tah...,"Ekspresi artistik individu, kolektif, atau kom...",https://data.go.id/dataset/dataset/data-organi...
4,Data Jumlah Surat Kabar di Kabupaten Mukomuko ...,"Surat kabar, atau koran, adalah publikasi peri...",https://data.go.id/dataset/dataset/data-jumlah...
...,...,...,...
95,DATA CAGAR BUDAYA DAN OBJEK DIDUGA CAGAR BUDAY...,DATA CAGAR BUDAYA DAN OBJEK DIDUGA CAGAR BUDAY...,https://data.go.id/dataset/dataset/data-cagar-...
96,Tenaga Pelestari Cagar Budaya yang dibina dan ...,- Cagar Budaya : cagar budaya adalah warisan ...,https://data.go.id/dataset/dataset/tenaga-pele...
97,Objek Cagar Budaya Yang Dilindungi,Data Sektoral Objek Cagar Budaya Yang Dilindun...,https://data.go.id/dataset/dataset/objek-cagar...
98,Apresiasi Budaya Lokal di Tingkat Nasional di ...,Apresiasi Budaya Lokal di Tingkat Nasional di ...,https://data.go.id/dataset/dataset/apresiasi-b...


In [7]:
# Path Menyimpan Dataset Process
tanggal_saat_ini = datetime.now()
slug_date = tanggal_saat_ini.strftime("%d-%m-%Y_%H-%M")
output_path = "../data/raw/dataset_keyword_satu_data" + slug_date + ".csv"
# print(output_path);

# Simpan data Process
df.to_csv(output_path, index=False)
print("Dataset berhasil disimpan:", output_path)

Dataset berhasil disimpan: ../data/raw/dataset_keyword_satu_data26-11-2026_16-51.csv
